# Lab 01: sprzedaż części samochodowych w Pythonie

W tym labie wczytasz dane o sprzedaży części samochodowych i przygotujesz ich podsumowanie. Dowiesz się, jak często poszczególne części się sprzedają i dla których miesięcy brakuje danych. Gotowe statystyki zapiszesz do pliku.

Skorzystamy ze zbioru Monash Car Parts. Zawiera on miesięczną sprzedaż 2 674 części od stycznia 1998 do marca 2002. W trzech na cztery miesiące sprzedaż wynosi zero. Długie przerwy między zakupami sprawiają, że taki popyt, nazywany przerywanym, trudno prognozować.

Na razie pracujemy bez pandas i NumPy. Użyjesz list, słowników, pętli oraz comprehensions, żeby samodzielnie odczytać dane i wykonać obliczenia. W module 06 zobaczysz, jak podobną pracę wykonać za pomocą pandas.

Zarezerwuj około 6 godzin. Możesz podzielić pracę na kilka sesji.

## Jak wykonać zadania

W większości zadań uzupełniasz funkcję w komórce notebooka. Jej nazwa, parametry i opis są już gotowe. Własny kod wpisz zamiast `raise NotImplementedError`.

Angielski opis pod `def`, zapisany w potrójnym cudzysłowie, to docstring. Określa, jakie dane funkcja otrzyma, co ma zwrócić i kiedy ma zgłosić błąd. Nad komórką znajdziesz te wymagania wyjaśnione po polsku.

Wywołanie `check("01.x", nazwa_funkcji)` sprawdza funkcję na małych, przykładowych danych. Znak ✅ oznacza, że sprawdzenie się powiodło. Jeśli zobaczysz ❌, przeczytaj komunikat: wskaże dane, dla których wynik nie zgadza się z oczekiwanym.

Wykonuj zadania po kolei i uruchamiaj komórki od góry. Funkcja z 01.2 korzysta z 01.1, a 01.3 z 01.2. Statystyki z 01.4 będą potrzebne do rankingu w 01.5 i zapisu w 01.6.

Jeśli utkniesz, rozwiń pierwszą podpowiedź i spróbuj ponownie. Kolejne otwieraj dopiero wtedy, gdy są potrzebne. Możesz też poprosić tutora o wskazówkę, na przykład `/hint 01.3`.

| Zadanie | Co przygotujesz | Orientacyjny czas |
|---|---|---|
| 01.1 | odczyt pojedynczej wartości, `parse_value` | 30 min |
| 01.2 | odczyt jednej linii, `parse_series_line` | 45 min |
| 01.3a | odczyt kolejnych serii, `read_series` | 45 min |
| 01.3b | słownik `series` i zbiór `lengths` | 15 min |
| 01.4a | statystyki jednej serii, `demand_stats` | 40 min |
| 01.4b | statystyki zbioru, `stats`, `missing_share`, `zero_share` | 20 min |
| 01.5 | ranking serii, `top_series` | 45 min |
| 01.6 | zapis statystyk, `save_summary` | 30 min |
| 01.7 | podział serii na grupy i odpowiedzi na dwa pytania | 45 min |


Uruchom poniższą komórkę, zanim przejdziesz do zadań. Przygotuje nazwy używane w dalszej części notebooka:

- `PATH` wskazuje plik z danymi pobrany poleceniem `uv run course data`. Jest obiektem `Path` z modułu `pathlib`, opisanego w sekcji „Pliki i `with`” wykładu 01.
- `paths` udostępnia ścieżki używane w kursie. W zadaniu 01.6 zapiszesz raport w folderze `paths.PROCESSED_DIR`.
- `check` uruchamia sprawdzenia Twoich funkcji.
- `json` pozwala zapisać słownik w formacie JSON. Użyjesz go w 01.6.


In [ ]:
import json
from pathlib import Path

from coursekit import paths
from coursekit.nb import check

PATH = paths.CAR_PARTS
print(PATH.name, f"{PATH.stat().st_size / 1000:.0f} kB")

## Obejrzyj plik z danymi

Zacznij od uruchomienia poniższego kodu. Wyświetli pierwszych 12 linii pliku, dzięki czemu zobaczysz, z jakim tekstem będziesz pracować. Blok `with` zadba o zamknięcie pliku po odczycie.


In [ ]:
with PATH.open(encoding="utf-8") as handle:
    for number, line in enumerate(handle):
        if number == 12:
            break
        print(line.rstrip()[:90])

Plik ma format `.tsf`. Na początku znajdują się komentarze oznaczone `#` oraz informacje o danych oznaczone `@`. Właściwe dane zaczynają się po linii `@data`.

Od tego miejsca każda linia opisuje sprzedaż jednej części. Tę historię jednej części nazywamy serią. Linia zawiera nazwę serii, datę początkową i wartości oddzielone przecinkami:

```text
nazwa:data startu:wartości oddzielone przecinkami
```

Pierwsza wartość dotyczy miesiąca wskazanego przez datę początkową. Każda następna dotyczy kolejnego miesiąca.

Znak `?` oznacza brak pomiaru. Nie zastępuj go zerem: zero mówi, że nic się nie sprzedało, a znak zapytania, że nie znamy wyniku.


## 01.1 Zamień tekst na liczbę

Po odczytaniu pliku Python widzi zapis `"3"` jako tekst. Żeby obliczyć sumę sprzedaży lub średnią, potrzebujemy liczby `3.0`. W danych pojawia się też `"?"`. Ten znak oznacza, że nie mamy pomiaru dla danego miesiąca.

Uzupełnij funkcję `parse_value` w komórce poniżej. Argument `text` zawiera pojedynczą wartość odczytaną z pliku. Funkcja ma zamienić ten tekst na liczbę typu `float`. Jeśli otrzyma znak `"?"`, ma zwrócić `None`.

Tak ma działać gotowa funkcja:

| Tekst przekazany do funkcji | Zwrócona wartość |
|---|---|
| `"3"` | `3.0` |
| `" 12 "` | `12.0` |
| `"2.5"` | `2.5` |
| `"0"` | `0.0` |
| `"?"` | `None` |

Spacje na początku i na końcu tekstu nie powinny wpływać na wynik. Dotyczy to również znaku `"?"`.

Zwróć uwagę na różnicę między zerem a brakiem danych. Zero oznacza, że w danym miesiącu nic się nie sprzedało. `None` oznacza, że nie wiemy, ile się sprzedało. Zastępowanie brakujących danych zerami może zaniżyć średnią sprzedaż.

### Kiedy zgłosić błąd

Nie każdy tekst da się zamienić na poprawną wartość sprzedaży. Funkcja ma zgłosić `ValueError` w dwóch przypadkach:

- otrzyma tekst, który nie jest liczbą ani znakiem `"?"`, na przykład `"abc"`;
- otrzyma tekst przedstawiający liczbę ujemną, na przykład `"-1"`. W tym zadaniu dopuszczamy tylko wartości nieujemne.

W komunikacie błędu umieść tekst, który spowodował problem. Dzięki temu osoba korzystająca z funkcji będzie wiedziała, co poprawić w danych.

### Gdzie zacząć

W komórce poniżej zastąp `raise NotImplementedError` własnym kodem. Nad funkcją znajdziesz stałą `MISSING`, której wartością jest `"?"`. Użyj jej do rozpoznawania braku danych.

Przydadzą Ci się metoda `strip()`, która usuwa spacje z końców tekstu, oraz funkcja `float()`, która zamienia tekst na liczbę. Obsługę `ValueError` znajdziesz w sekcji "Wyjątki" wykładu 01. Jeśli potrzebujesz więcej wskazówek, rozwiń podpowiedzi poniżej.

Po uzupełnieniu funkcji uruchom komórkę. Wywołanie `check("01.1", parse_value)` sprawdzi poprawne wartości, brak danych i błędne dane. Zielony znak ✅ oznacza, że zadanie jest wykonane poprawnie.

<details>
<summary>Podpowiedź 1: kierunek</summary>

`float("3")` zamienia tekst na liczbę, a dla tekstu, który liczbą nie jest, rzuca `ValueError`. Spacje usuwa `text.strip()`.

</details>

<details>
<summary>Podpowiedź 2: podejście</summary>

Najpierw przypadek szczególny: `"?"`. Potem `try` z `float(...)` i `except ValueError`, w którym rzucasz własny `ValueError` z lepszym komunikatem. Na końcu sprawdzenie znaku.

</details>

<details>
<summary>Podpowiedź 3: prawie kod</summary>

```python
text = text.strip()
if text == MISSING:
    return None
try:
    value = float(text)
except ValueError as error:
    raise ValueError(f"Not a number: {text!r}") from error
# teraz: ujemna wartość to też ValueError
```

</details>

In [ ]:
MISSING = "?"


def parse_value(text):
    """Turn one value of a .tsf series into a float, or None for a missing month.

    Args:
        text: One value cut out of a data line, a string such as "3", " 12 ",
            "2.5" or "?". Spaces around it are allowed.

    Returns:
        The demand of one month as a float ("3" gives 3.0), or None when the
        text is "?" (MISSING), which marks a month without a measurement.

    Raises:
        ValueError: The text is not a number, or the number is negative.
            The message shows the text.
    """
    raise NotImplementedError("Zadanie 01.1")


check("01.1", parse_value)

## 01.2 Odczytaj historię jednej części

Potrafisz już zamienić pojedynczy tekst na liczbę lub `None`. Teraz odczytasz całą linię, czyli historię sprzedaży jednej części.

Uzupełnij funkcję `parse_series_line(line)`. Argument `line` zawiera tekst w takiej postaci:

```text
T7:1998-01-01 00-00-00:0,2,?,1
```

Funkcja ma zwrócić trzy informacje w krotce, w tej kolejności:

| Informacja | Wynik dla przykładowej linii |
|---|---|
| nazwa serii | `"T7"` |
| data początkowa bez godziny | `"1998-01-01"` |
| lista miesięcznych wartości | `[0.0, 2.0, None, 1.0]` |

Cały wynik to `("T7", "1998-01-01", [0.0, 2.0, None, 1.0])`. Używamy krotki, bo te trzy elementy razem opisują jeden rekord, a każdy ma inne znaczenie.

### Wykorzystaj poprzednią funkcję

Do zamiany kolejnych wartości użyj `parse_value` z zadania 01.1. Nie musisz jej importować: wystarczy wcześniej uruchomić komórkę z jej definicją. Jeśli `parse_value` zgłosi błąd, pozwól mu przerwać również `parse_series_line`.

Na końcu odczytanej linii może znajdować się znak nowej linii `\n`. Usuń białe znaki z obu końców tekstu za pomocą `strip()`.

Metoda `split(":", 2)` dzieli tekst najwyżej przy dwóch dwukropkach. W ten sposób otrzymasz trzy części. Jeśli części jest mniej, zgłoś `ValueError`. Z daty usuń godzinę, a tekst z wartościami podziel przy przecinkach. Do utworzenia listy wyników możesz użyć list comprehension.

Po uzupełnieniu funkcji uruchom komórkę. Sprawdzenie `check("01.2", parse_series_line)` sprawdzi odczyt przykładowej linii ze znakiem `\n` na końcu oraz odrzucenie linii `"T7 0,1,2"`, w której brakuje dwukropków.

<details>
<summary>Podpowiedź 1: kierunek</summary>

`text.split(":", 2)` dzieli tekst najwyżej dwa razy, więc trzecia część zostaje w całości, nawet jeśli ma w sobie dwukropki. Data i godzina oddzielone są spacją.

</details>

<details>
<summary>Podpowiedź 2: podejście</summary>

Po podziale sprawdź, czy są trzy części. Datę bierzesz z drugiej części przed spacją. Wartości to trzecia część podzielona przecinkiem, a każdy kawałek przechodzi przez `parse_value`. List comprehension zrobi to w jednej linii.

</details>

<details>
<summary>Podpowiedź 3: prawie kod</summary>

```python
parts = line.strip().split(":", 2)
if len(parts) != 3:
    raise ValueError(...)
name, start, values = parts
return name, start.split(" ")[0], [...]
```

</details>

In [ ]:
def parse_series_line(line):
    """Split one data line into (name, start date, list of values).

    Args:
        line: One line after "@data", a string such as
            "T7:1998-01-01 00-00-00:0,2,?,1". A trailing newline is allowed.

    Returns:
        A tuple of three fields: the series name ("T7"), the start date
        without the time ("1998-01-01") and the list of monthly values, each
        turned by parse_value into a float or None ([0.0, 2.0, None, 1.0]).

    Raises:
        ValueError: The line does not have the form name:start:values, or a
            value is not a valid demand (see parse_value).
    """
    raise NotImplementedError("Zadanie 01.2")


check("01.2", parse_series_line)

## 01.3 Wczytaj serie z pliku

Teraz połączysz dotychczasowe funkcje, żeby odczytać cały plik. Najpierw napiszesz funkcję, która udostępnia kolejne serie. Potem zbierzesz je w słowniku.

### 01.3a Odczytaj jedną serię naraz

Uzupełnij funkcję `read_series(path)`. Argument `path` wskazuje plik `.tsf`. Może być obiektem `Path`, takim jak `PATH`, albo tekstem zawierającym ścieżkę.

Funkcja ma być generatorem. Zamiast tworzyć listę wszystkich wyników, generator udostępnia je po jednym, za pomocą `yield`. Pętla, która z niego korzysta, pobiera następny wynik dopiero wtedy, gdy jest potrzebny. Więcej przykładów znajdziesz w sekcji „Iteracja i generatory” wykładu 01.

W tym zadaniu każdy wynik to krotka zwrócona przez `parse_series_line`: nazwa, data początkowa i lista wartości. Zachowaj kolejność serii z pliku.

### Które linie odczytać

Pomiń komentarze zaczynające się od `#` i wszystkie puste linie. Pomiń też nagłówek: serie zaczynają się dopiero po linii `@data`. Samej linii `@data` nie przekazuj do `parse_series_line`.

Przyda Ci się zmienna pamiętająca, czy linia `@data` została już odczytana. Otwórz plik przez `with`, użyj kodowania `utf-8` i przechodź po liniach pętlą. Białe znaki z końców każdej linii usuń metodą `strip()`.

Taki odczyt nie wymaga przechowywania całego tekstu pliku w pamięci. W danej chwili przetwarzasz jedną linię i udostępniasz jedną serię.

Uruchom `check("01.3", read_series)`. To sprawdzenie dotyczy części 01.3a. Upewnia się, że funkcja jest generatorem i z małego pliku zawierającego nagłówek, komentarze oraz trzy serie udostępnia tylko te serie.

<details>
<summary>Podpowiedź 1: kierunek</summary>

`with path.open(encoding="utf-8") as handle:` i pętla `for line in handle:`. W środku pętli `yield`, zamiast dopisywania do listy.

</details>

<details>
<summary>Podpowiedź 2: podejście</summary>

Potrzebujesz zmiennej, która pamięta, czy jesteś już za `@data`. Na początku `False`, a gdy trafisz na linię `@data`, ustawiasz `True`. Pusta linia i linia zaczynająca się od `#` to `continue`.

</details>

<details>
<summary>Podpowiedź 3: prawie kod</summary>

```python
in_data = False
with Path(path).open(encoding="utf-8") as handle:
    for line in handle:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if in_data:
            yield ...
        elif line == "@data":
            in_data = True
```

</details>

In [ ]:
def read_series(path):
    """Yield (name, start, values) for every series of a .tsf file.

    Args:
        path: Location of the .tsf file, a pathlib.Path or a string.

    Yields:
        One tuple per data line, in file order, as returned by
        parse_series_line: (name, start date, list of float or None).
        Comments (#), header lines (@) up to and including "@data", and
        blank lines are skipped.

    Raises:
        ValueError: A data line cannot be parsed (see parse_series_line).
    """
    raise NotImplementedError("Zadanie 01.3")


check("01.3", read_series)

### 01.3b Zbierz serie w słowniku

Generator pozwala przejść po wynikach tylko raz. Do kolejnych zadań potrzebujemy danych, do których można wracać, dlatego zapisz je w słowniku.

Uzupełnij dwie zmienne w komórce poniżej:

- W `series` zapisz słownik, w którym kluczem jest nazwa serii, a wartością lista jej miesięcznych pomiarów. Na przykład pod kluczem `"T1"` znajdą się liczby i wartości `None` dotyczące tej części. Datę początkową możesz tutaj pominąć.
- W `lengths` zapisz zbiór długości tych list. Każda długość pojawi się w zbiorze tylko raz, więc zobaczysz, czy wszystkie serie obejmują tyle samo miesięcy.

Do utworzenia `series` użyj `read_series(PATH)` i dict comprehension. Następnie przejdź po `series.values()` za pomocą set comprehension, żeby utworzyć `lengths`.

Uruchom komórkę oraz znajdujące się pod nią instrukcje `assert`. Liczba serii powinna wynosić 2 674, zgodnie z opisem zbioru we wstępie.


In [ ]:
series = ...
lengths = ...
print(f"Serie: {len(series)}, długości: {lengths}")

In [ ]:
assert isinstance(series, dict), "series ma być słownikiem: nazwa serii -> lista wartości."
assert isinstance(lengths, set), "lengths ma być zbiorem długości list."
assert len(series) == 2674, "W pliku jest 2 674 serii. Czy pomijasz nagłówek i puste linie?"

## 01.4 Podsumuj sprzedaż

Dane są już w pamięci. Teraz obliczysz statystyki jednej serii, a potem wykorzystasz tę samą funkcję dla wszystkich części.

### 01.4a Oblicz statystyki jednej serii

Uzupełnij funkcję `demand_stats(values)`. Otrzyma listę miesięcznych pomiarów, czyli liczb typu `float` i wartości `None`. Nie zmieniaj tej listy.

Funkcja ma zwrócić słownik z pięcioma polami:

| Klucz | Co obliczyć | Typ wyniku |
|---|---|---|
| `months` | liczba miesięcy ze znanym pomiarem, w tym zerowym | `int` |
| `missing` | liczba miesięcy z `None` | `int` |
| `total` | suma znanych wartości sprzedaży | `float` |
| `zero_share` | liczba znanych miesięcy z zerową sprzedażą podzielona przez liczbę wszystkich znanych miesięcy | `float` lub `None` |
| `adi` | liczba znanych miesięcy podzielona przez liczbę miesięcy ze sprzedażą większą od zera | `float` lub `None` |

Jeśli nie ma znanych miesięcy, `zero_share` ma być równe `None`. Jeśli nie ma ani jednego miesiąca z dodatnią sprzedażą, `adi` też ma być równe `None`. W obu sytuacjach unikamy dzielenia przez zero.

### Co mówi ADI

ADI, z angielskiego *average demand interval*, opisuje przeciętny odstęp między miesiącami z popytem. W tym zadaniu obliczamy je ze znanych wartości sprzedaży. ADI równe 1 oznacza sprzedaż w każdym znanym miesiącu, a ADI równe 4 sprzedaż średnio w jednym na cztery znane miesiące.

Dla listy `[0.0, 2.0, None, 0.0, 6.0]` oczekujemy:

```python
{"months": 4, "missing": 1, "total": 8.0, "zero_share": 0.5, "adi": 2.0}
```

Zacznij od wybrania znanych wartości, a następnie tych większych od zera. Możesz użyć comprehensions z wykładu 01. Pamiętaj, że warunek `if v` odrzuciłby również zera. Do pomijania braków służy `if v is not None`.

Po uzupełnieniu funkcji uruchom `check("01.4", demand_stats)`. Sprawdzenie dotyczy części 01.4a. Obejmuje powyższy przykład, listę `[0.0, None, 0.0]` bez dodatniej sprzedaży i listę `[None, None]` bez znanych pomiarów.

<details>
<summary>Podpowiedź 1: kierunek</summary>

Zacznij od dwóch list z comprehensions: znane wartości i wartości większe od zera. Resztę liczysz z ich długości i sum.

</details>

<details>
<summary>Podpowiedź 2: podejście</summary>

Uważaj na dzielenie przez zero. Seria z samymi brakami ma zero znanych miesięcy, seria z samymi zerami ma zero miesięcy z popytem. W obu przypadkach odpowiednie pole to `None`. Warunek `if v is not None`, nie `if v`: zero to prawdziwa obserwacja.

</details>

<details>
<summary>Podpowiedź 3: prawie kod</summary>

```python
known = [v for v in values if v is not None]
demand = [v for v in known if v > 0]
return {
    "months": len(known),
    "missing": ...,
    "total": sum(known),
    "zero_share": ... if known else None,
    "adi": ... if demand else None,
}
```

</details>

In [ ]:
def demand_stats(values):
    """Summarise one series: known months, missing months, total, zero share, ADI.

    Args:
        values: The monthly values of one series in month order, a list of
            float or None (None marks a missing month). It is not changed.

    Returns:
        A dict with five keys:
        "months": number of known months (int);
        "missing": number of missing months (int);
        "total": sum of the known values, the units sold;
        "zero_share": share of known months with zero demand, from 0 to 1,
        or None when no month is known;
        "adi": average demand interval, known months divided by months with
        demand above zero, or None when there was no demand.
    """
    raise NotImplementedError("Zadanie 01.4")


check("01.4", demand_stats)

### 01.4b Oblicz udział braków i zer w całym zbiorze

Wiesz już, jak podsumować jedną serię. Teraz sprawdzisz, jak dużo braków danych i zerowej sprzedaży występuje we wszystkich seriach razem.

W komórce poniżej uzupełnij trzy zmienne:

- `stats` to słownik statystyk wszystkich serii. Pod nazwą każdej serii zapisz wynik `demand_stats` dla jej listy z `series`. Z tego słownika skorzystasz również w zadaniach 01.5 i 01.6.
- `missing_share` to liczba brakujących pomiarów ze wszystkich serii podzielona przez liczbę wszystkich miesięcy, zarówno znanych, jak i brakujących.
- `zero_share` to liczba miesięcy z zerową sprzedażą ze wszystkich serii podzielona przez liczbę wszystkich znanych miesięcy.

Oba udziały zapisz jako liczby typu `float` od 0 do 1.

### Policz miesiące, zamiast uśredniać udziały

Nie obliczaj średniej z `zero_share` poszczególnych serii. Serie mogą mieć różną liczbę znanych miesięcy. Każdy miesiąc ma się liczyć tak samo, więc najpierw zsumuj liczbę miesięcy z zerową sprzedażą i liczbę znanych miesięcy. Dopiero potem podziel te sumy.

Liczbę zer dla jednej serii odtworzysz jako `zero_share * months`. Jeśli dana seria nie ma znanych pomiarów, jej `zero_share` wynosi `None`. Pomiń ją w tym sumowaniu: nie dodaje ani zer, ani znanych miesięcy. W tym pliku nie ma takich serii, ale kod powinien działać również wtedy, gdy się pojawią.

Do sumowania wartości ze słownika `stats` możesz użyć `sum()` z comprehension.

Po uruchomieniu komórki udział zer powinien być zbliżony do wartości ze wstępu: trzy na cztery znane miesiące. Uruchom też komórkę z instrukcjami `assert`.


In [ ]:
stats = ...
missing_share = ...
zero_share = ...
print(f"Braki: {missing_share:.1%} komórek, zera: {zero_share:.1%} znanych miesięcy")

In [ ]:
assert isinstance(stats, dict) and len(stats) == len(series), "stats: jeden wpis na serię."
assert 0 < missing_share < 1, "missing_share to udział: liczba od 0 do 1."
assert 0 < zero_share < 1, "zero_share to udział: liczba od 0 do 1."

## 01.5 Porównaj serie w rankingu

Które części sprzedały się w największej ilości? Które sprzedają się najrzadziej? Przygotujesz funkcję, która wybierze serie o najwyższej wartości wskazanej statystyki.

Uzupełnij `top_series(stats, key, n)`. Argumenty oznaczają:

- `stats` to słownik z zadania 01.4b. Pod nazwą każdej serii zawiera słownik jej statystyk. Nie zmieniaj go.
- `key` to nazwa statystyki, na przykład `"total"` lub `"adi"`.
- `n` to maksymalna liczba serii w wyniku.

Funkcja ma zwrócić listę par `(nazwa, wartość)`, od największej wartości do najmniejszej. Pomiń serie, dla których wybrana statystyka jest równa `None`. Gdy dwie wartości są równe, ustaw nazwy alfabetycznie. Jeśli pasujących serii jest mniej niż `n`, zwróć krótszą listę.

Na przykład dla danych `{"T1": {"total": 5.0}, "T2": {"total": 9.0}, "T3": {"total": 9.0}}`, statystyki `key="total"` i liczby `n=2` wynik to `[("T2", 9.0), ("T3", 9.0)]`.

### Jak uporządkować wynik

Przyda Ci się `sorted()` z funkcją `lambda`, omówione w części „Sortowanie po kluczu” wykładu 01. Zwróć uwagę na dwie różne rzeczy o nazwie `key`: w Twojej funkcji jest to tekst z nazwą statystyki, natomiast argument `key` w `sorted()` przyjmuje funkcję określającą kolejność.

Zbuduj nową listę par, pomijając brakujące wartości. Uporządkuj ją według wymaganej kolejności. Pierwsze `n` elementów wybierzesz wycinkiem `[:n]`.

Sprawdzenie `check("01.5", top_series)` obejmuje remisy, pomijanie `None` i zachowanie oryginalnego słownika bez zmian. Kolejna komórka pokaże rankingi dla danych o częściach samochodowych.

<details>
<summary>Podpowiedź 1: kierunek</summary>

`sorted(...)` przyjmuje argument `key`: funkcję, która dla elementu zwraca to, po czym sortujesz. Jeśli zwróci krotkę, Python porówna najpierw pierwsze pola, potem drugie.

</details>

<details>
<summary>Podpowiedź 2: podejście</summary>

Zbuduj listę par z pominięciem `None`, posortuj rosnąco po `(-wartość, nazwa)` i weź pierwsze `n` wycinkiem `[:n]`. Minus przed wartością daje kolejność malejącą, a nazwa rozstrzyga remisy rosnąco.

</details>

<details>
<summary>Podpowiedź 3: prawie kod</summary>

```python
rows = [(name, s[key]) for name, s in stats.items() if s[key] is not None]
return sorted(rows, key=lambda row: ...)[:n]
```

</details>

In [ ]:
def top_series(stats, key, n):
    """Return the n (name, value) pairs with the largest `key`, ties by name.

    Args:
        stats: Statistics of many series, a dict from series name to the
            dict returned by demand_stats. It is not changed.
        key: Name of the statistic to rank by, such as "total" or "adi".
        n: How many series to return at most.

    Returns:
        A list of at most n tuples (name, value), largest value first; equal
        values in ascending name order. Series whose `key` is None are left
        out, so the list is shorter than n when fewer series have a value.
    """
    raise NotImplementedError("Zadanie 01.5")


check("01.5", top_series)

In [ ]:
print("Najwięcej sprzedanych sztuk:", top_series(stats, "total", 5))
print("Najrzadszy popyt (ADI):     ", top_series(stats, "adi", 5))

## 01.6 Zapisz statystyki w pliku

Dotychczasowe wyniki znajdują się w pamięci kernela. Po jego zatrzymaniu trzeba byłoby obliczyć je ponownie. Zapisz statystyki w pliku JSON, żeby móc do nich wrócić i odczytać je w innym programie.

Uzupełnij funkcję `save_summary(stats, path)`. Przekażesz jej słownik z zadania 01.4b oraz miejsce zapisu. Argument `path` może być obiektem `Path` albo tekstem zawierającym ścieżkę. Słownika nie zmieniaj.

Gotowa funkcja ma zapisać plik i zwrócić jego ścieżkę jako obiekt `Path`. Zastosuj kodowanie UTF-8, wcięcia o szerokości dwóch spacji i alfabetyczną kolejność kluczy. Dzięki temu plik będzie czytelny, a porównanie dwóch wersji nie pokaże różnic wynikających tylko z kolejności pól.

Jeśli folder docelowy lub któryś z jego folderów nadrzędnych nie istnieje, utwórz go. Jeśli plik już istnieje, nadpisz go.

### Z czego skorzystać

Moduły `json` i `Path` są już dostępne po uruchomieniu pierwszej komórki. Najpierw zamień otrzymaną ścieżkę na obiekt `Path`. Następnie utwórz potrzebne foldery, zamień słownik na tekst JSON i zapisz ten tekst. Na końcu zwróć ścieżkę.

Przykład zapisu innego słownika znajdziesz w części „Zapis do pliku JSON”, w sekcji „Pliki i `with`” wykładu 01.

Uruchom `check("01.6", save_summary)`. Następna komórka zapisze Twoje statystyki do `data/processed/car_parts_summary.json`. Otwórz plik w edytorze i sprawdź, czy widzisz nazwy serii i obliczone wartości.

<details>
<summary>Podpowiedź 1: kierunek</summary>

Moduł `json` ma `json.dumps(obiekt, indent=..., sort_keys=...)`. `Path` ma `.parent.mkdir(...)` i `.write_text(...)`.

</details>

<details>
<summary>Podpowiedź 2: podejście</summary>

`mkdir(parents=True, exist_ok=True)` tworzy wszystkie brakujące katalogi i nie narzeka, gdy już są. Zapisz tekst z kodowaniem UTF-8.

</details>

<details>
<summary>Podpowiedź 3: prawie kod</summary>

```python
path = Path(path)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(stats, indent=2, sort_keys=True), encoding="utf-8")
return path
```

</details>

In [ ]:
def save_summary(stats, path):
    """Write the statistics as sorted, indented JSON and return the file path.

    Args:
        stats: The dict to save, from series name to its statistics. It is
            not changed.
        path: Where to write the file, a pathlib.Path or a string. Missing
            folders are created; an existing file is overwritten.

    Returns:
        The path of the written file as a pathlib.Path. The file is UTF-8
        JSON with an indent of 2 spaces and keys sorted alphabetically.
    """
    raise NotImplementedError("Zadanie 01.6")


check("01.6", save_summary)

In [ ]:
written = save_summary(stats, paths.PROCESSED_DIR / "car_parts_summary.json")
print(f"Zapisano {written.relative_to(paths.ROOT)}")

## 01.7 Zinterpretuj wyniki

Na koniec wykorzystasz statystyki, żeby odpowiedzieć na dwa pytania. Ta część nie ma automatycznego sprawdzenia. Pierwsza odpowiedź wymaga kodu i krótkiego komentarza, a druga samego wyjaśnienia.

### Pytanie 1: jak często pojawia się sprzedaż?

Jednym ze sposobów podziału serii jest użycie ADI. Serie z ADI nie większym niż 1.32 nazywamy regularnymi, a te z ADI większym niż 1.32 przerywanymi.

Korzystając ze słownika `stats`, utwórz dwie listy nazw serii:

- w `regular` umieść nazwy serii z `adi <= 1.32`;
- w `intermittent` umieść nazwy serii z `adi > 1.32`.

Jeśli `adi` jest równe `None`, nie umieszczaj serii w żadnej z list. Do ich utworzenia możesz użyć list comprehension z warunkiem.

Wyświetl liczebność obu grup. Pod kodem napisz jedno zdanie: która grupa przeważa i czy zgadza się to z opisem zbioru we wstępie?


In [ ]:
regular = ...
intermittent = ...
print(f"Regularne: {len(regular)}, przerywane: {len(intermittent)}")

*Twoja odpowiedź na pytanie 1:*

### Pytanie 2: czy mały błąd prognozy wystarcza magazynowi?

Wyobraź sobie regułę: „W następnym miesiącu sprzeda się tyle samo co w poprzednim”. Dla rzadko sprzedawanej części taka prognoza często wynosi zero.

Oceńmy ją przez średnią wielkość pomyłki w sztukach. Dla każdego miesiąca obliczamy różnicę między prognozą a sprzedażą, pomijamy znak i uśredniamy te wartości.

Dlaczego ta prognoza może mieć mały średni błąd, a mimo to nie pomagać w zaopatrzeniu magazynu? Pomyśl o miesiącu, w którym klient chce kupić część i towar musi być dostępny. Odpowiedz słowami, bez obliczeń.


*Twoja odpowiedź na pytanie 2:*

## Zanim zakończysz

Zrestartuj kernel i uruchom wszystkie komórki od początku, wybierając Restart i Run All. Upewnij się, że każde sprawdzenie kończy się znakiem ✅. Ten lab sprawdzasz z notebooka, ponieważ Twoje funkcje znajdują się w jego komórkach.

Zapisz działający notebook. W module 02 przeniesiesz te same sześć funkcji wraz z docstringami do pliku `carparts.py`. Następnie dodasz opisy typów i sprawdzisz kod narzędziami używanymi w pracy.
